**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/markowitz-2023"], check=True)
    os.chdir("/content/encore/entries/markowitz-2023")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Markowitz et al. (2023): the earlier attempt's findings, checked on dandiset 000559 version 0.261113.1911

Paper: Markowitz, Gillis, Jay et al. (2023), "Spontaneous behaviour is structured by reinforcement
without explicit reward", Nature 614, 108-117, doi:10.1038/s41586-022-05611-2.

Dataset: DANDI:000559, version 0.261113.1911, the version this entry's other notebooks use. The
files are streamed; nothing is downloaded to disk.

This notebook belongs to the reconciliation of attempt `markowitz-2023-02` with the earlier attempt
`markowitz-2023-01`, made on version 0.260528.0858 (docs/protocol.md, "Repeat attempts"). It checks,
on this version, each earlier finding that the new attempt did not itself report. It tests no claim
of the paper. Each section opens with the earlier finding's statement.

| earlier finding | here |
|---|---|
| 1. catch trials not marked | checked |
| 2. genotype and opsin not stated | checked |
| 3. session type and number not stored | checked |
| 4. 635-nm stimulator in every closed-loop file | checked |
| 5. dLight-only mice described as DAT-IRES-Cre and Ai32 | checked |
| 6. syllable timestamps not ascending | checked |
| 7. repeated syllable frames in two files of mouse 3155 | checked |
| 8. empty syllable series in 14 dLight+ChrimsonR files | checked |
| 9. dLight+ChrimsonR 2-s and pulsed sessions missing, fewer DLS experiments | reported by the new attempt (its findings 1 and 3) |
| 10. food-reward sessions missing | reported by the new attempt (its finding 1) |
| 11. behaviour-only files and failed sessions not flagged | reported by the new attempt (its findings 2 and 6) |
| 12. re-referenced, z-scored dLight trace not stored | checked |

In [ ]:
import numpy as np
import pandas as pd
import markowitz as M

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)
print("dandiset", M.DANDISET, "version", M.VERSION)


def _clock(f, key):
    """Order of a syllable series' timestamps: non-ascending and backward steps, the largest
    backward step (s and as a share of the series' span), repeated timestamps, and the median
    interval of the distinct timestamps."""
    if key + "/timestamps" not in f:
        return dict(present=False)
    t = f[key + "/timestamps"][()]
    n = len(t)
    if n < 2:
        return dict(present=True, n=n)
    dt = np.diff(t)
    back = dt < 0
    u = np.unique(t)
    span = float(t.max() - t.min())
    return dict(present=True, n=n, n_nonasc=int((dt <= 0).sum()), n_back=int(back.sum()),
                max_back_s=float(-dt[back].min()) if back.any() else 0.0,
                back_frac=float(-dt[back].min() / span) if back.any() and span > 0 else 0.0,
                repeated=int(n - len(u)), dt_med_unique=float(np.median(np.diff(u))) if len(u) > 1 else np.nan)


def survey(row):
    """The metadata, tables and syllable clocks of one file that the earlier findings are about."""
    with M._open(row["asset_id"]) as f:
        s = lambda p: M._s(f[p][()]) if p in f else None
        d = dict(path=row["path"], kind=row["kind"], subject=s("general/subject/subject_id"),
                 genotype=s("general/subject/genotype"), strain=s("general/subject/strain"),
                 subj_desc=s("general/subject/description"), session_description=s("session_description"))
        g = f["general"]
        d["devices"] = {k: M._s(v.attrs.get("description", "")) for k, v in g["devices"].items()} if "devices" in g else {}
        site = "general/optogenetics/OptogeneticStimulusSite"
        d["exc_nm"] = float(f[site + "/excitation_lambda"][()]) if site in f else np.nan
        d["site_location"] = s(site + "/location")
        ogs = "stimulus/presentation/OptogeneticSeries"
        d["n_light_on"] = int((f[ogs + "/data"][()] > 0).sum()) if ogs in f else 0
        b = f["processing/behavior"] if "processing/behavior" in f else {}
        d["has_td"] = "TargetDetections" in b
        if d["has_td"]:
            st = b["TargetDetections/stimulated"][()].astype(bool)
            d["td_n"], d["td_stim"] = len(st), int(st.sum())
        d["has_ts"] = "TargetSessions" in b
        if d["has_ts"]:
            ts = b["TargetSessions"]
            d["ts_cols"] = tuple(sorted(k for k in ts if k != "id"))
            for k in ["session_number", "experiment_type", "exclude"]:
                d["ts_" + k] = tuple(M._s(x) for x in ts[k][()]) if k in ts else None
        for which in ["Offline", "Online"]:
            for k, v in _clock(f, "processing/behavior/BehavioralSyllable" + which).items():
                d[which.lower() + "_" + k] = v
        d["processing"] = {m: tuple(sorted(f["processing"][m].keys())) for m in f["processing"]} if "processing" in f else {}
        d["analysis"] = tuple(sorted(f["analysis"].keys())) if "analysis" in f else ()
        d["acquisition"] = tuple(sorted(f["acquisition"].keys())) if "acquisition" in f else ()
    return d


def survey_retry(row):
    for k in range(3):
        try:
            return survey(row)
        except OSError:
            if k == 2:
                raise

In [ ]:
assets = M.list_assets()
S = pd.DataFrame(M.load_many(survey_retry, [r.to_dict() for _, r in assets.iterrows()], procs=16))
S["group"] = S.subj_desc.str.extract(r"Lab group code: ([^.]+)\.")[0].fillna("(none)")
S["stim"] = S.n_light_on > 0
S["photometry"] = S.processing.apply(lambda p: "SignalDfOverF" in p.get("ophys", ()))
print("NWB files read:", len(S), "of", len(assets), "; subjects:", S.subject.nunique())
print()
print("files by kind and group:")
print(S.groupby(["kind", "group"], dropna=False).size().to_string())
print()
print("files with light pulses (OptogeneticSeries > 0) by kind:", S[S.stim].groupby("kind").size().to_dict())
print("files with a dLight dF/F series by kind:", S[S.photometry].groupby("kind").size().to_dict())

## Earlier finding 1: catch trials

> The closed-loop files mark stimulation onsets but not catch trials (target detections on which stimulation was withheld); the conversion kept only the stimulated detections. Comparisons of stimulation with catch trials (Fig. 3i, 3l, 3m, 4f) cannot be made, and a substitute selection shows stimulation-catch differences in 9 of 12 no-opsin controls.

In [ ]:
cl = S[S.kind.isin(["reinforcement", "reinforcement-photometry"])]
print("closed-loop files by stimulation and TargetDetections table:")
print(cl.groupby(["kind", "stim", "has_td"]).size().to_string())
x = cl[cl.stim & cl.has_td]
catch = x.td_n - x.td_stim
print()
print("stimulation files with a TargetDetections table:", len(x))
print("  detections %d, stimulated %d, not stimulated (catch) %d" % (x.td_n.sum(), x.td_stim.sum(), catch.sum()))
print("  files with at least one catch detection: %d of %d" % ((catch > 0).sum(), len(x)))
print("  share stimulated per file: median %.3f, range %.3f to %.3f" % ((x.td_stim / x.td_n).median(), (x.td_stim / x.td_n).min(), (x.td_stim / x.td_n).max()))
print("  files whose stimulated detections equal their light onsets: %d of %d" % ((x.td_stim == x.n_light_on).sum(), len(x)))

**Result:** the closed-loop files of this version hold a TargetDetections table with a `stimulated`
column, and the detections without stimulation (catch trials) are marked; the counts are printed
above.

## Earlier finding 2: genotype and opsin

> No file states the genotype or opsin of a mouse. Opto-DA (Ai32) and no-opsin control mice are told apart only by the stimulation site ("snc (axon)" or "ctrl"), and the dLight+ChrimsonR mice only by their subject names.

In [ ]:
sub = S.groupby("subject").agg(group=("group", "first"), genotype=("genotype", lambda v: " | ".join(sorted(set(map(str, v))))),
                               strain=("strain", lambda v: " | ".join(sorted(set(map(str, v))))),
                               n_descriptions=("subj_desc", "nunique"))
print("subjects with more than one subject description across their files:", int((sub.n_descriptions > 1).sum()), "of", len(sub))
print()
print("subjects by lab group code, genotype and strain:")
print(sub.groupby(["group", "genotype", "strain"]).size().to_string())
print()
for g, d in S.groupby("group").subj_desc.first().items():
    print(g, "::", d)

**Result:** every mouse's files state a genotype, and the subject description names the opsin or
its absence; one mouse of the snc-eyfp group says its genotype is not given in the source.

## Earlier finding 3: session type and number

> Session type (baseline, stimulation, post) and session number are not stored; they must be inferred from free-text descriptions, dates and stimulation pulses. Five files with stimulation are described as recording sessions.

In [ ]:
print("files with a TargetSessions table by kind:", S[S.has_ts].groupby("kind").size().to_dict(),
      "; without:", S[~S.has_ts].groupby("kind").size().to_dict())
print("TargetSessions columns:", S.ts_cols.dropna().value_counts().to_dict())
T = S[S.has_ts].copy()
T["numbers"] = T.ts_session_number.apply(lambda v: tuple(sorted(set(float(x) for x in v))))
T["stim_number"] = T.numbers.apply(lambda v: any(x in (1.0, 2.0) for x in v))
print()
print("files by kind, light pulses and whether TargetSessions gives a stimulation number (1 or 2):")
print(T[T.kind != "photometry"].groupby(["kind", "stim", "stim_number"]).size().to_string())
rec = T[T.stim & T.session_description.str.lower().str.startswith("recording")]
print()
print("files with light pulses whose session description says recording:", len(rec))
print(rec[["subject", "kind", "session_description", "n_light_on", "ts_session_number", "ts_experiment_type"]].to_string())
odd = T[~T.stim & T.stim_number & (T.kind != "photometry")]
print()
print("files without light pulses whose TargetSessions gives a stimulation number:", len(odd))
print(odd[["subject", "kind", "session_description", "ts_session_number", "ts_experiment_type"]].to_string())

**Result:** every mouse file but the 5 keypoint sessions holds a TargetSessions table with the
session number and experiment type, and the 5 files with light pulses that are described as
recording sessions carry stimulation numbers (1 or 2) there. 23 files without light pulses also
carry a stimulation number (10 dLight+ChrimsonR habituation files, 7 velocity-modulation files and
6 reinforcement files that are a stimulation day of one target and a baseline day of another); they
are listed above.

## Earlier finding 4: stimulator wavelength

> Every closed-loop file states a 635-nm stimulator (Opto Engine MRL-III-635), including the Ai32 (ChR2) mice, which the Methods say were stimulated with 470-nm light.

In [ ]:
S["stimulator"] = S.devices.apply(lambda d: "; ".join(f"{k}: {v}" for k, v in sorted(d.items())
                                                   if k != "kinect" and "camera" not in k))
o = S[S.kind.isin(["reinforcement", "reinforcement-photometry", "velocity-modulation"])]
print("closed-loop and velocity-modulation files by group, stated excitation wavelength and stimulator:")
print(o.groupby(["group", "exc_nm", "stimulator"]).size().to_string())

**Result:** the Ai32 and no-opsin control files state 470 nm and a 470-nm light source; only the
dLight+ChrimsonR files state 635 nm and the Opto Engine MRL-III-635.

## Earlier finding 5: subject description of the dLight-only mice

> The dLight-only mice (C57BL/6J in the Methods) carry the subject description "wild-type DAT-IRES-Cre and Ai32 mice", the same as the optogenetic mice.

In [ ]:
d = S[S.group.isin(["dls-dlight", "dms-dlight"])]
print(d.groupby(["group", "genotype", "strain", "subj_desc"]).size().to_string())
print("files of these groups whose description names Ai32:", int(d.subj_desc.str.contains("Ai32").sum()), "of", len(d))

**Result:** the dLight-only mice of DLS and DMS are described as C57BL/6J, wild type, with no opsin,
and none of their files names Ai32.

## Earlier finding 6: syllable timestamps out of order

> In 28 photometry and 547 closed-loop files the syllable timestamps are not ascending. In 573 of these 575 the rows are rotated, with one backward step, which is at least 0.95 of the session length in 558 files and 0.635 of it at the least. Sorting by timestamp restores a regular 30-Hz clock (median interval 0.03337 s in every file).

In [ ]:
for w in ["offline", "online"]:
    p = S[S[w + "_present"].fillna(False).astype(bool)]
    print(f"{w} syllable series: {len(p)} files ({p.groupby('kind').size().to_dict()})")
    print(f"  files with a non-ascending step: {int((p[w + '_n_nonasc'] > 0).sum())}; with a backward step: {int((p[w + '_n_back'] > 0).sum())}")
    print(f"  median interval of distinct timestamps: {p[w + '_dt_med_unique'].min():.5f} to {p[w + '_dt_med_unique'].max():.5f} s")

**Result:** in no file of this version are the offline or online syllable timestamps out of order,
and every series has a median interval of 0.03337 s.

## Earlier finding 7: repeated syllable frames

> Two closed-loop files of mouse 3155 store every syllable frame twice (53,822 and 53,862 repeated timestamps), and have two and three non-ascending steps.

In [ ]:
m = S[S.subject == "3155"]
print("files of mouse 3155:", len(m))
print("  online series: repeated timestamps in %d files, non-ascending steps in %d files" % ((m.online_repeated > 0).sum(), (m.online_n_nonasc > 0).sum()))
print("  online rows per file: %d to %d; files with 53,822 or 53,862 rows: %d" % (m.online_n.min(), m.online_n.max(), m.online_n.isin([53822, 53862]).sum()))
print("files of any mouse with repeated timestamps: offline %d, online %d" % ((S.offline_repeated > 0).sum(), (S.online_repeated > 0).sum()))

**Result:** no file of mouse 3155, or of any mouse, has repeated syllable timestamps; the 3155 files
hold one row per frame (the row counts are printed above).

## Earlier finding 8: empty syllable series

> 14 files of dLight+ChrimsonR mice hold empty syllable series; all list two target syllables.

In [ ]:
c = S[S.group == "dls-chrimson-dlight"]
print("dLight+ChrimsonR files:", len(c))
for w in ["offline", "online"]:
    print(f"  {w} series present in {int(c[w + '_present'].sum())}; with fewer than 2 rows: {int((c[w + '_n'] < 2).sum())}; rows {int(c[w + '_n'].min())} to {int(c[w + '_n'].max())}")
print("files of any kind with an empty or one-row syllable series:", int(((S.offline_n < 2) | (S.online_n < 2)).sum()))

**Result:** every dLight+ChrimsonR file holds offline and online syllable series of 35,545 to
53,902 rows; no file of the dandiset has an empty one.

## Earlier finding 12: re-referenced dLight trace

> The re-referenced, z-scored dLight trace used by every photometry analysis is not stored; it must be recomputed from the stored dF/F and smoothed reference, whose regression fit is not stored either.

In [ ]:
p = S[S.photometry]
print("files with a dLight dF/F series:", len(p))
print("their processing modules and contents:")
print(p.processing.apply(lambda d: "; ".join(f"{k}: {', '.join(v)}" for k, v in sorted(d.items()) if k != "behavior")).value_counts().to_string())
print("their analysis groups:", p.analysis.value_counts().to_dict())
print("their acquisition groups:", p.acquisition.value_counts().to_dict())
print("files of any kind with a series whose name mentions z-scor or reref:",
      int(S.processing.apply(lambda d: any(("zscor" in k.lower() or "reref" in k.lower()) for v in d.values() for k in v)).sum()))

**Result:** the photometry files hold the dLight and reference dF/F, the smoothed references and the
raw photometry, and no re-referenced or z-scored trace or the fit behind one; the trace has still to
be recomputed.

## Summary

| earlier finding | on version 0.261113.1911 |
|---|---|
| 1. catch trials not marked | fixed: TargetDetections marks the unstimulated detections |
| 2. genotype and opsin not stated | fixed: genotypes and opsins stated; one snc-eyfp mouse's genotype is said not to be in the source |
| 3. session type and number not stored | fixed: TargetSessions stores them |
| 4. 635-nm stimulator in every closed-loop file | fixed: 470 nm for the Ai32 and control mice |
| 5. dLight-only mice described as DAT-IRES-Cre and Ai32 | fixed: described as C57BL/6J |
| 6. syllable timestamps not ascending | fixed |
| 7. repeated syllable frames, mouse 3155 | fixed |
| 8. empty syllable series | fixed |
| 12. re-referenced, z-scored trace not stored | still present |